# Strands on AgentCore: Production Deploy + Feature Ownership (TravelMind)

**Track:** Agentic AI Bootcamp &nbsp;|&nbsp; **Level:** Advanced

Read `05_strands_with_agentcore.md` first. Here we build the **both** patterns in code:
- Strands loop + **AgentCore Memory** via a hook (the key integration)
- Strands MCP client + **AgentCore Gateway** (pattern)
- **AgentCore Code Interpreter** called through a Strands tool
- **StrandsTelemetry** feeding AgentCore Observability
- Wrapped in a **Runtime** entrypoint and deployed

Mental model: **Strands owns behavior, AgentCore owns operations.**


## Strands owns behavior, AgentCore owns operations

```mermaid
flowchart TD
    subgraph Strands[Strands: behavior]
        LOOP[agent loop]
        TOOLS[tool definitions]
        PROMPT[prompt, model choice]
    end
    subgraph AgentCore[AgentCore: operations]
        RT[Runtime: host]
        MEM[Memory: durable state]
        CI[Code Interpreter]
        OBS[Observability]
    end
    Strands -->|runs on| AgentCore
```


## 0. Setup

**VS Code:** venv as kernel, `aws configure` (`us-east-1`), `pip install bedrock-agentcore strands-agents strands-agents-tools boto3`.
**Colab:** `pip install ...` first cell; credentials via secrets/env. `app.run()` is best in a terminal.
**Model:** `us.anthropic.claude-haiku-4-5-20251001-v1:0` enabled.


In [ ]:
%pip install -q --upgrade "boto3>=1.39.9" bedrock-agentcore strands-agents strands-agents-tools

---
## 1. Both-pattern #1: Strands loop + AgentCore Memory (hook)

Strands runs the loop and stays stateless per process. A **hook** bridges to AgentCore Memory: load recent turns on start, write each message back. This is the most useful integration in the framework.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memc = MemoryClient(region_name=REGION)
stm = memc.create_memory_and_wait(
    name=f"tm-strands-stm-{uuid.uuid4().hex[:8]}", strategies=[], event_expiry_days=7)
MEM_ID = stm["id"]
ACTOR, SESSION = PASSENGER, f"pnr-{PNR}-{uuid.uuid4().hex[:8]}"
print("memory:", MEM_ID, "| session:", SESSION)


In [ ]:
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent

class ShortTermMemoryHook(HookProvider):
    """Load recent turns on init; persist each message as it is added."""
    def __init__(self, client, memory_id, actor_id, session_id):
        self.client = client
        self.memory_id = memory_id
        self.actor_id = actor_id
        self.session_id = session_id

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_start)
        registry.add_callback(MessageAddedEvent, self.on_message)

    def on_start(self, event):
        try:
            turns = self.client.get_last_k_turns(
                memory_id=self.memory_id, actor_id=self.actor_id,
                session_id=self.session_id, k=4)
            if turns:
                # surface prior context to the agent (strategy is up to you;
                # here we prepend a compact recap to the system prompt)
                recap = "Prior context:\n" + json.dumps(turns, default=str)[:1500]
                event.agent.system_prompt += "\n\n" + recap
                print(f"[hook] loaded {len(turns)} prior turns")
        except Exception as e:
            print("[hook] load skipped:", e)

    def on_message(self, event):
        try:
            msgs = getattr(event.agent, "messages", [])
            if not msgs:
                return
            last = msgs[-1]
            role = str(last.get("role", "assistant")).upper()
            content = last.get("content", "")
            text = content if isinstance(content, str) else json.dumps(content, default=str)
            self.client.create_event(
                memory_id=self.memory_id, actor_id=self.actor_id,
                session_id=self.session_id, messages=[(text[:4000], role)])
        except Exception as e:
            print("[hook] save skipped:", e)

print("hook defined")


## 2. AgentCore Code Interpreter, called through a Strands tool

Strands defines the tool; AgentCore runs the code. This is a "Strands calls AgentCore" case, not either/or.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.tools.code_interpreter_client import code_session

@tool
def get_pnr(pnr: str) -> str:
    """Look up a booking by PNR."""
    db = {"JX48Q2": {"passenger": "Rao", "tier": "Gold", "segment": "BLR-DEL",
                     "status": "CANCELLED", "base_fare": 8200, "taxes": 640}}
    return json.dumps(db.get(pnr, {"error": "not found"}))

@tool
def compute_refund(base_fare: float, taxes: float, tier: str) -> str:
    """Compute refund + tier bonus by running code in the AgentCore sandbox."""
    bonus_pct = 10 if tier.lower() == "gold" else 0
    src = f"r={base_fare}+{taxes}\nb=round(r*{bonus_pct}/100,2)\nprint({{'refund':r,'bonus':b,'total':r+b}})"
    with code_session(REGION) as c:
        resp = c.invoke("executeCode", {"language": "python", "code": src, "clearContext": False})
        out = ""
        for ev in resp["stream"]:
            for it in ev["result"].get("content", []):
                if it.get("type") == "text":
                    out += it["text"]
        return out

print("tools ready")


## 3. StrandsTelemetry -> AgentCore Observability

One line: Strands emits OTEL spans; AgentCore/CloudWatch collects them (after the one-time Transaction Search enable). In a bare notebook without a collector this is a no-op; on Runtime it lights up the GenAI Observability page.

In [ ]:
try:
    from strands.telemetry import StrandsTelemetry
    StrandsTelemetry().setup_otlp_exporter()
    print("OTLP exporter set (spans flow to CloudWatch when hosted on Runtime)")
except Exception as e:
    print("telemetry setup note:", e)


## 4. Assemble the agent and test locally

Behavior (loop, tools, prompt) is Strands. Operations (memory, code sandbox, telemetry) are AgentCore, plugged in.

In [ ]:
hook = ShortTermMemoryHook(memc, MEM_ID, ACTOR, SESSION)

travelmind = Agent(
    model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
    tools=[get_pnr, compute_refund],
    hooks=[hook],
    system_prompt=("You are TravelMind, an airline support agent. "
                   "Use get_pnr to look up bookings and compute_refund for refund math. "
                   "Be concise."),
)

# first turn: agent looks up PNR, computes refund; hook writes turns to memory
r1 = travelmind("PNR JX48Q2 BLR-DEL got cancelled. What's my Gold-tier refund credit?")
print(r1.message)


In [ ]:
# second turn, same session: the hook already persisted the first exchange
r2 = travelmind("Remind me which segment was cancelled.")
print(r2.message)

# confirm memory captured the conversation
turns = memc.get_last_k_turns(memory_id=MEM_ID, actor_id=ACTOR, session_id=SESSION, k=6)
print("\nstored turns:", len(turns))


## 5. Both-pattern #2: Strands MCP client + AgentCore Gateway (pattern)

In production the tools come from Gateway over MCP, not from local `@tool`s. Strands is the MCP client. Needs a real gateway URL + bearer token (see the features notebook to create a gateway).

In [ ]:
# PATTERN (needs a real gateway URL + OAuth token)
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.streamable_http import streamablehttp_client

GATEWAY_MCP_URL = None      # e.g. "https://<id>.gateway.bedrock-agentcore.us-east-1.amazonaws.com/mcp"
ACCESS_TOKEN    = None      # OAuth bearer for inbound auth

if GATEWAY_MCP_URL and ACCESS_TOKEN:
    gateway = MCPClient(lambda: streamablehttp_client(
        GATEWAY_MCP_URL, headers={"Authorization": f"Bearer {ACCESS_TOKEN}"}))
    with gateway:
        mcp_tools = gateway.list_tools_sync()   # e.g. TravelOps___get_pnr
        prod_agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
            tools=mcp_tools,
            system_prompt="You are TravelMind. Use the provided tools.")
        print(prod_agent("Look up PNR JX48Q2").message)
else:
    print("Set GATEWAY_MCP_URL + ACCESS_TOKEN to run against a real gateway.")


## 6. Wrap for Runtime and deploy

The same agent, hosted. Note the build/serve split and streaming entrypoint.

In [ ]:
deploy_file = '''
import json, uuid
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.memory import MemoryClient
from strands.telemetry import StrandsTelemetry
from strands.hooks import HookProvider, HookRegistry, MessageAddedEvent, AgentInitializedEvent
import os

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
REGION   = "us-east-1"
MEM_ID   = os.getenv("MEMORY_ID")            # inject via env / config, do not hardcode

StrandsTelemetry().setup_otlp_exporter()     # observability
memc = MemoryClient(region_name=REGION)

@tool
def get_pnr(pnr: str) -> str:
    db = {"JX48Q2": {"passenger":"Rao","tier":"Gold","segment":"BLR-DEL","status":"CANCELLED"}}
    return json.dumps(db.get(pnr, {"error":"not found"}))

# ... (ShortTermMemoryHook as in section 1) ...

app = BedrockAgentCoreApp()
# build time
_base_agent = Agent(model=BedrockModel(model_id=MODEL_ID, region_name=REGION),
                    tools=[get_pnr],
                    system_prompt="You are TravelMind. Be concise and accurate.")

@app.entrypoint
async def invoke(payload, context):
    # context.session_id (>=16 chars) correlates memory + traces
    prompt = payload.get("prompt", "")
    async for event in _base_agent.stream_async(prompt):
        yield event

if __name__ == "__main__":
    app.run()
'''
with open("travelmind_runtime.py", "w") as f:
    f.write(deploy_file)
print("wrote travelmind_runtime.py")


**Deploy (terminal):**

```bash
# new CLI (recommended)
npm install -g @aws/agentcore
agentcore create --defaults           # framework Strands
agentcore deploy
agentcore invoke --prompt "PNR JX48Q2 refund?" --session-id "$(uuidgen)" --stream

# legacy toolkit
pip install bedrock-agentcore-starter-toolkit
agentcore configure -e travelmind_runtime.py
agentcore launch
agentcore invoke '{"prompt": "PNR JX48Q2 refund?"}'
```

Invoke from boto3 after deploy:
```python
rt = boto3.client("bedrock-agentcore", region_name="us-east-1")
resp = rt.invoke_agent_runtime(agentRuntimeArn="<arn>",
                               payload=json.dumps({"prompt": "..."}).encode())
```

**Production notes (post-cell):**
- Inject `MEMORY_ID` and region via env/config, never hardcode.
- Use IAM roles, not access keys. IAM action is `bedrock:InvokeModel` (not `bedrock:Converse`).
- Tools that touch authenticated services -> Gateway + Identity, secrets never in tool code.
- Add retries/timeouts around tool + model calls; return a useful message on failure.


---
## 7. Cleanup

In [ ]:
if globals().get("MEM_ID"):
    try: memc.delete_memory(memory_id=MEM_ID); print("deleted memory", MEM_ID)
    except Exception as e: print("mem:", e)
# DeleteAgentRuntime for any runtime you deployed, and delete any gateway/lambda you created.


**Next:** the same production question for LangChain / LangGraph / LangSmith, where LangGraph owns graph state + checkpointing and LangSmith owns tracing, and AgentCore provides hosting + memory bridges. See `07_langchain_langgraph_langsmith_with_agentcore.md` + notebook.